In [20]:
# extract point from localization map

In [10]:
from pathlib import Path
import sys

# adjust this depending on where the notebook sits
# example: notebook is in project/notebooks/experiments/
PROJECT_ROOT = Path.cwd().resolve().parents[0]   # go up 2 levels

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT: \n{PROJECT_ROOT}")

import yaml
from pathlib import Path
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import torch.nn.functional as F
import cv2
from tqdm import tqdm
import numpy as np

from datasets.bcdata import BCDataDataset, collate_heatmap_points
from datasets.transforms import PointsToLocalizationHeatmap, PointsToCountHeatmap


from visualization import overlay_heatmap


from models.models import HybridModel
from models.losses import weighted_sigmoid_mse_from_logits, softplus_mse_from_logits, l1_count_from_density_logits

from src.debug import print_info
#from src.evaluation import count_metrics
from src.points_v2 import heatmaps_to_points_batch


import torch
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

PROJECT_ROOT: 
/raid/home/user6/projects/IHC/BCData_Ki67


In [11]:
with open("../config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

data_root = Path(cfg["h200_paths"]["data_root"])
checkpoint_dir = Path(cfg["h200_paths"]["checkpoint_dir"])
print(f"data_root: {data_root}")
print(f"checkpoint_dir: {checkpoint_dir}")

data_root: /raid/datasets/Yeldos/BCData
checkpoint_dir: checkpoints


In [12]:
loc_heatmap_generator = PointsToLocalizationHeatmap(out_hw=(160,160), in_hw=(640,640), sigma=2.0)
count_heatmap_generator = PointsToCountHeatmap(out_hw=(160,160), in_hw=(640,640), sigma=2.0)

train_dataset = BCDataDataset(root = data_root,
                        split="train",
                        target_loc_transform = loc_heatmap_generator,
                        target_count_transform = count_heatmap_generator)


test_dataset = BCDataDataset(root = data_root,
                        split="test",
                        target_loc_transform = loc_heatmap_generator,
                        target_count_transform = count_heatmap_generator)

val_dataset = BCDataDataset(root = data_root,
                        split="validation",
                        target_loc_transform = loc_heatmap_generator,
                        target_count_transform = count_heatmap_generator)

In [13]:
model = HybridModel()
device = 'cuda'
model.to(device);



model.load_state_dict(torch.load("../checkpoints/hybrid_02.pt"))
model.eval();

# Проверяем алгоритм на GT хитпамах

In [24]:

for i in range(100):
    img, loc_heatmap, count_heatmap, pos_pts, neg_pts = train_dataset[i]
    img = img.to(device)
    loc_heatmap = loc_heatmap.to(device)
    count_heatmap = count_heatmap.to(device)


    pred_loc_hm, pred_den_hm, pred_count = model(img.unsqueeze(0))


    #print_info(pos_pts, "pos_pts")
    #print_info(neg_pts, "neg_pts")

    out_pos, out_neg = heatmaps_to_points_batch(loc_heatmap.unsqueeze(0), kernel_size=5, threshold = 0.9)

    if len(pos_pts) != len(out_pos[0]):
        print(f"{i}:  {len(pos_pts), len(out_pos[0])}")
    if len(neg_pts) != len(out_neg[0]):
        print(f"{i}:  {len(neg_pts), len(out_neg[0])}")


23:  (69, 67)
35:  (73, 72)
53:  (81, 80)
64:  (14, 13)
77:  (293, 292)
93:  (50, 49)


# Проверяем на хитмапах модели

In [44]:
for i in range(100):
    img, loc_heatmap, count_heatmap, pos_pts, neg_pts = test_dataset[i]
    img = img.to(device)
    loc_heatmap = loc_heatmap.to(device)
    count_heatmap = count_heatmap.to(device)


    pred_loc_hm, pred_den_hm, pred_count = model(img.unsqueeze(0))
    out_pos, out_neg = heatmaps_to_points_batch(pred_loc_hm, kernel_size=5, threshold = 0.9)
    print(f"({len(pos_pts)}, {len(out_pos[0])};    ({len(neg_pts)}, {len(out_neg[0])})")

(32, 31;    (106, 52)
(19, 15;    (90, 69)
(45, 30;    (53, 27)
(143, 105;    (126, 108)
(108, 105;    (38, 35)
(100, 89;    (53, 38)
(116, 97;    (35, 30)
(110, 114;    (10, 8)
(81, 82;    (36, 28)
(49, 46;    (26, 18)
(54, 61;    (22, 15)
(73, 76;    (15, 11)
(104, 106;    (29, 22)
(29, 26;    (65, 46)
(93, 75;    (25, 18)
(162, 135;    (51, 48)
(141, 87;    (109, 68)
(189, 128;    (97, 78)
(51, 42;    (71, 57)
(63, 44;    (63, 35)
(46, 32;    (78, 60)
(53, 45;    (115, 94)
(53, 37;    (121, 88)
(58, 40;    (108, 85)
(70, 58;    (82, 43)
(48, 40;    (124, 95)
(41, 38;    (122, 97)


KeyboardInterrupt: 